# 02 — Compare BM25, semantic, and hybrid retrieval

This notebook uses the same generated chunks and manually labelled questions as the command-line evaluation scripts. Its purpose is to make the results inspectable: you can read the evidence, not merely a metric.

The sequence is:

`question → retrieve chunks → compare with manual labels → calculate Recall@k`

No LLM generates an answer in this notebook. That is intentional: retrieval quality must be understood before grounded generation is added.

## 1. Load chunks and evaluation labels

Only answerable questions with at least one `relevant_chunk_id` are scoreable in a retrieval metric. Unanswerable questions belong to the later abstention evaluation.

In [1]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from askml_rag.models import Chunk, EvaluationQuestion

In [2]:
def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the AskML RAG project root.")

In [3]:
PROJECT_ROOT = find_project_root()
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks" / "chunks.jsonl"
QUESTIONS_PATH = PROJECT_ROOT / "data" / "evaluation" / "questions.yaml"

chunks = [
    Chunk.model_validate_json(line)
    for line in CHUNKS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

payload = yaml.safe_load(QUESTIONS_PATH.read_text(encoding="utf-8"))
questions = [
    EvaluationQuestion.model_validate(question) for question in payload["questions"]
]

scoreable_questions = [
    question
    for question in questions
    if question.answerable and question.relevant_chunk_ids
]

print(f"Chunks loaded: {len(chunks)}")
print(f"Total benchmark questions: {len(questions)}")
print(f"Scoreable answerable questions: {len(scoreable_questions)}")

Chunks loaded: 28
Total benchmark questions: 8
Scoreable answerable questions: 5


In [4]:
pd.DataFrame(
    {
        "question_id": question.question_id,
        "answerable": question.answerable,
        "category": question.category.value,
        "labelled_chunks": len(question.relevant_chunk_ids),
        "question": question.question,
    }
    for question in questions
)

,question_id,answerable,category,labelled_chunks,question
0,recommendation_experience_001,True,direct_fact,1,What experience does Marcelo have with recomme...
1,work_atmosphere_001,True,direct_fact,3,What does Marcelo expect from a work atmosphere?
2,python_experience_001,True,direct_fact,3,What Python experience does Marcelo have?
3,deep_learning_experience_001,True,direct_fact,3,What deep-learning experience does Marcelo have?
4,kubernetes_experience_001,False,unanswerable,0,What experience does Marcelo have with Kuberne...
5,terraform_experience_001,False,unanswerable,0,What experience does Marcelo have with Terraform?
6,gcp_services_001,True,unanswerable,3,Which GCP services are documented in Marcelo's...
7,unsupported_computing_cost_001,False,unanswerable,0,What was the exact computing cost of the TOROS...


## 2. Build the three retrievers

- **BM25** rewards exact shared terms.
- **Semantic retrieval** compares embedding vectors and can identify related wording.
- **Hybrid retrieval** applies Reciprocal Rank Fusion (RRF), promoting chunks ranked highly by both methods.

The model is cached after the first download.

In [5]:
from sentence_transformers import SentenceTransformer

from askml_rag.retrieval.bm25 import BM25Retriever
from askml_rag.retrieval.hybrid import HybridRetriever
from askml_rag.retrieval.semantic import SemanticRetriever

/home/mlares/store/Learning/AI_engineering/askml_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

bm25 = BM25Retriever(chunks)
embedder = SentenceTransformer(MODEL_NAME)
semantic = SemanticRetriever(chunks, embedder)
hybrid = HybridRetriever(bm25, semantic, candidate_limit=10)

print(f"Embedding model: {MODEL_NAME}")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3257.01it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 3. Inspect one question interactively

Change `question_id` to any labelled question. Read the retrieved text before judging a method. A matching ID is useful, but the actual question is whether the chunk supports the desired claim.

In [7]:
questions_by_id = {question.question_id: question for question in questions}
question_id = scoreable_questions[0].question_id
question = questions_by_id[question_id]
K = 3

display(Markdown(f"## {question.question_id}"))
display(Markdown(f"**Question:** {question.question}"))
display(Markdown(f"**Manually labelled evidence:** `{question.relevant_chunk_ids}`"))

## recommendation_experience_001

**Question:** What experience does Marcelo have with recommendation systems?

**Manually labelled evidence:** `['skills_chunk_000']`

In [8]:
def inspect_results(name: str, results: list[Chunk]) -> None:
    display(Markdown(f"### {name}"))
    table = pd.DataFrame(
        {
            "rank": rank,
            "chunk_id": chunk.chunk_id,
            "document_id": chunk.document_id,
            "is_labelled_evidence": chunk.chunk_id in question.relevant_chunk_ids,
            "preview": chunk.text.replace("\n", " ")[:280],
        }
        for rank, chunk in enumerate(results, start=1)
    )
    display(table)


inspect_results("BM25", bm25.search(question.question, limit=K))
inspect_results("Semantic", semantic.search(question.question, limit=K))
inspect_results("Hybrid RRF", hybrid.search(question.question, limit=K))

### BM25

,rank,chunk_id,document_id,is_labelled_evidence,preview
0,1,website_home_chunk_000,website_home,False,Applied AI technical leadership · data science...
1,2,skills_chunk_000,skills,True,# Marcelo Lares — Skills Review and Retrieval ...
2,3,skills_chunk_001,skills,False,Recommendation Systems \* Recommender Systems ...


### Semantic

,rank,chunk_id,document_id,is_labelled_evidence,preview
0,1,skills_chunk_004,skills,False,Risk \* Agriculture \* SmartFarm \* Tourism \*...
1,2,skills_chunk_000,skills,True,# Marcelo Lares — Skills Review and Retrieval ...
2,3,website_home_chunk_000,website_home,False,Applied AI technical leadership · data science...


### Hybrid RRF

,rank,chunk_id,document_id,is_labelled_evidence,preview
0,1,website_home_chunk_000,website_home,False,Applied AI technical leadership · data science...
1,2,skills_chunk_000,skills,True,# Marcelo Lares — Skills Review and Retrieval ...
2,3,skills_chunk_001,skills,False,Recommendation Systems \* Recommender Systems ...


## 4. Evaluate the complete labelled benchmark

Recall@k is `1` when at least one manually labelled evidence chunk appears in the first `k` results; otherwise it is `0`. On a small benchmark it is a diagnostic, not a final claim that one method is superior.

In [9]:
from askml_rag.evaluation.retrieval import recall_at_k

retrievers = {
    "BM25": bm25,
    "Semantic": semantic,
    "Hybrid RRF": hybrid,
}

records = []
for name, retriever in retrievers.items():
    for benchmark_question in scoreable_questions:
        results = retriever.search(benchmark_question.question, limit=K)
        records.append(
            {
                "method": name,
                "question_id": benchmark_question.question_id,
                "recall_at_k": recall_at_k(benchmark_question, results, k=K),
                "retrieved_chunk_ids": [chunk.chunk_id for chunk in results],
                "expected_chunk_ids": benchmark_question.relevant_chunk_ids,
            }
        )

evaluation = pd.DataFrame(records)
summary = evaluation.groupby("method", as_index=False).agg(
    questions=("question_id", "count"),
    mean_recall_at_k=("recall_at_k", "mean"),
)
summary

,method,questions,mean_recall_at_k
0,BM25,5,0.8
1,Hybrid RRF,5,1.0
2,Semantic,5,1.0


In [10]:
evaluation.sort_values(["question_id", "method"])

,method,question_id,recall_at_k,retrieved_chunk_ids,expected_chunk_ids
3,BM25,deep_learning_experience_001,1.0,"[website_home_chunk_000, skills_chunk_000, ski...","[skills_chunk_000, skills_chunk_001, website_h..."
13,Hybrid RRF,deep_learning_experience_001,1.0,"[skills_chunk_000, website_home_chunk_000, ski...","[skills_chunk_000, skills_chunk_001, website_h..."
8,Semantic,deep_learning_experience_001,1.0,"[skills_chunk_000, skills_chunk_004, website_p...","[skills_chunk_000, skills_chunk_001, website_h..."
4,BM25,gcp_services_001,0.0,"[website_projects_chunk_011, website_projects_...","[skills_chunk_000, skills_chunk_002, skills_ch..."
14,Hybrid RRF,gcp_services_001,1.0,"[website_research_chunk_000, skills_chunk_003,...","[skills_chunk_000, skills_chunk_002, skills_ch..."
9,Semantic,gcp_services_001,1.0,"[skills_chunk_004, skills_chunk_003, skills_ch...","[skills_chunk_000, skills_chunk_002, skills_ch..."
2,BM25,python_experience_001,1.0,"[website_home_chunk_000, skills_chunk_000, web...","[skills_chunk_000, website_home_chunk_000, web..."
12,Hybrid RRF,python_experience_001,1.0,"[skills_chunk_000, skills_chunk_004, website_h...","[skills_chunk_000, website_home_chunk_000, web..."
7,Semantic,python_experience_001,1.0,"[skills_chunk_000, skills_chunk_004, website_p...","[skills_chunk_000, website_home_chunk_000, web..."
0,BM25,recommendation_experience_001,1.0,"[website_home_chunk_000, skills_chunk_000, ski...",[skills_chunk_000]


## Questions to explore

1. Rewrite a question using synonyms. Which method changes most?
2. Set `K` to 1, 3, and 5. Does a method find evidence, but rank it too low?
3. For every `0` score, decide whether the issue is the retrieval method, chunking, or the manual label.
4. Add a new independently written question to `questions.yaml`; do not derive it from the retrieved text.

Next: open `03_embedding_visualization.ipynb` to see the geometric representation used by semantic retrieval.